# AI Governance Agent - Data Pipeline
**Meridian Governance Group | AAI-510 Final Project**

This notebook builds the data pipeline for the AI Policy Research Agent. It downloads three authoritative AI governance documents, extracts and chunks the text, and loads it into Databricks Vector Search for retrieval.

## Documents
- **NIST AI RMF 1.0** - AI Risk Management Framework
- **NIST AI 600-1** - Generative AI Profile
- **EU AI Act** - EU Regulation 2024/1689

In [0]:
# Install required libraries
%pip install pypdf langchain langchain-community

## Step 1: Download Source Documents
Download the three PDFs directly from their official sources.

In [0]:
import urllib.request

# Some websites block automated downloads unless the request looks like a browser
# Adding a 'User-Agent' header tricks the server into allowing the download
opener = urllib.request.build_opener()
opener.addheaders = [('User-Agent', 'Mozilla/5.0')]
urllib.request.install_opener(opener)

urls = {
    "nist_ai_600_1": "https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.600-1.pdf",
    "eu_ai_act": "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=OJ:L_202401689",
    "nist_ai_rmf": "https://nvlpubs.nist.gov/nistpubs/ai/nist.ai.100-1.pdf"
}

for name, url in urls.items():
    urllib.request.urlretrieve(url, f"/tmp/{name}.pdf")
    print(f"Downloaded {name}")

## Step 2: Extract and Chunk Text
Parse text from each PDF and split into 1,000 character chunks with 200 character overlap to preserve context at boundaries.

In [0]:
from pypdf import PdfReader

# Loop through each PDF we downloaded and extract all the text
for name in urls.keys():
    # Open the PDF file from where we saved it
    reader = PdfReader(f"/tmp/{name}.pdf")
    
    # Count how many pages it has
    print(f"{name}: {len(reader.pages)} pages")

In [0]:
from pypdf import PdfReader

# Dictionary to store the extracted text from each document
documents = {}

for name in urls.keys():
    reader = PdfReader(f"/tmp/{name}.pdf")
    
    # Extract text from every page and join it into one big string
    full_text = ""
    for page in reader.pages:
        full_text += page.extract_text()
    
    # Store it in our dictionary
    documents[name] = full_text
    print(f"{name}: {len(full_text)} characters extracted")

In [0]:
%pip install langchain-text-splitters

In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# chunk_size: how many characters per chunk
# chunk_overlap: how many characters overlap between chunks so we don't lose context at the edges
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# List to hold all chunks across all 3 documents
all_chunks = []

for name, text in documents.items():
    # Split this document's text into chunks
    chunks = splitter.split_text(text)
    
    # For each chunk, store the text and which document it came from
    for chunk in chunks:
        all_chunks.append({
            "source": name,
            "text": chunk
        })

print(f"Total chunks created: {len(all_chunks)}")
print(f"Example chunk:\n{all_chunks[0]['text'][:300]}")

## Step 3: Store in Delta Table
Save all 1,107 chunks to a Delta table in Unity Catalog with Change Data Feed enabled for Vector Search sync.

In [0]:
from pyspark.sql import Row

# Convert our list of chunks into a Spark DataFrame
# Databricks works natively with Spark, so this is the best way to store data here
rows = [Row(source=chunk["source"], text=chunk["text"]) for chunk in all_chunks]
df = spark.createDataFrame(rows)

# Save it as a Delta table in your catalog
df.write.mode("overwrite").saveAsTable("main.default.ai_governance_chunks")

print(f"Saved {df.count()} chunks to Delta table")

In [0]:
# Read the table back to confirm everything saved correctly
df_check = spark.sql("SELECT * FROM main.default.ai_governance_chunks LIMIT 5")
df_check.show(truncate=80)

## Step 4: Create Vector Search Index
Create a Vector Search endpoint and index using BGE large embeddings to enable semantic similarity search over the documents.

In [0]:
%pip install databricks-vectorsearch

In [0]:
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()
print("Vector Search client created successfully")

In [0]:
# Create a Vector Search endpoint - this is the server that will host our index
# This can take a few minutes to provision
client.create_endpoint(
    name="ai_governance_endpoint",
    endpoint_type="STANDARD"
)
print("Endpoint created")

In [0]:
# Drop the existing table
spark.sql("DROP TABLE IF EXISTS main.default.ai_governance_chunks")

from pyspark.sql import Row
from pyspark.sql.functions import monotonically_increasing_id

# Recreate from our all_chunks list with id column this time
rows = [Row(source=chunk["source"], text=chunk["text"]) for chunk in all_chunks]
df = spark.createDataFrame(rows)
df_with_id = df.withColumn("id", monotonically_increasing_id())

# Save with Change Data Feed enabled from the start
df_with_id.write.mode("overwrite").option("delta.enableChangeDataFeed", "true").saveAsTable("main.default.ai_governance_chunks")

# Verify
spark.sql("DESCRIBE TABLE main.default.ai_governance_chunks").show()

In [0]:
# Create the Vector Search index on our chunks table
# source_table: the Delta table we want to index
# pipeline_type: TRIGGERED means we manually sync it, CONTINUOUS would auto-sync
# primary_key: unique identifier for each row -- we need to add one first
index = client.create_delta_sync_index(
    endpoint_name="ai_governance_endpoint",
    index_name="main.default.ai_governance_index",
    source_table_name="main.default.ai_governance_chunks",
    pipeline_type="TRIGGERED",
    primary_key="id",
    embedding_source_column="text",
    embedding_model_endpoint_name="databricks-bge-large-en"
)
print("Index created")

In [0]:
import time

# Wait for the index to finish syncing before we can query it
print("Waiting for index to sync...")
while True:
    index = client.get_index("ai_governance_endpoint", "main.default.ai_governance_index")
    status = index.describe()["status"]["ready"]
    if status:
        print("Index is ready!")
        break
    print("Still syncing...")
    time.sleep(30)

## Step 5: Validate the Pipeline
Run a test query to confirm the index is returning relevant results.

In [0]:
from databricks.vector_search.client import VectorSearchClient
client = VectorSearchClient()
print("Client ready")

# Test the index with a sample query relevant to our use case
index = client.get_index("ai_governance_endpoint", "main.default.ai_governance_index")

results = index.similarity_search(
    query_text="What are the requirements for high risk AI systems?",
    columns=["source", "text"],
    num_results=3
)

for r in results["result"]["data_array"]:
    print(f"Source: {r[0]}")
    print(f"Text: {r[1][:200]}")
    print("---")